In [1]:
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# a set of libraries that perhaps should always be in Python source
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import os
import socket
import sys
import getopt
import inspect
import warnings
import json
import pickle
from pathlib import Path
import itertools
import datetime
import re
import shutil
import string
import json
import glob
import random
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Additional libraries for this work
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import math
from base64 import b64decode
from IPython.display import Image
import requests

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Data Science Libraries
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import numpy as np
import pandas as pd

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# Graphics
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import matplotlib.pyplot as plt
from PIL import Image
import PIL.ImageOps

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# import the OpenAI Python library for calling the OpenAI API
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import openai

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# XML/JSON support for DoD STIGs
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
import xml.etree.ElementTree as ET
from stig_parser import convert_stig

# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
# NLP work
# +++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
from nltk import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer


# Input Sources

In [2]:
###########################################
#- CORE PATHS
###########################################
DATA_DIR = "/projects/data/stig/zip"
SAVE_DIR = "/projects/data/stig/train"

try:
    if not ( os.path.exists(DATA_DIR) ):
        os.mkdir(DATA_DIR)
except Exception as e:
    print(f"ERROR detected trying to create the root DATA path as follows: {str(e)}")

###########################################
#- SAVE FILES
###########################################
pcpf_save_filename = SAVE_DIR + os.sep + "pcpf_stig_llm_promptinput.json"
gpt35_save_filename = SAVE_DIR + os.sep + "gpt35_stig_llm_promptinput.json"

pcpf_train_save_filename=SAVE_DIR+os.sep+"train_pcpf.json"
pcpf_test_save_filename=SAVE_DIR+os.sep+"test_pcpf.json"

gpt35_train_save_filename=SAVE_DIR+os.sep+"train_gpt35.json"
gpt35_test_save_filename=SAVE_DIR+os.sep+"test_gpt35.json"

###########################################
#- PROMPT INPUTS
###########################################
PROMPT_PRE_SYSTEM="You are an AI assistant named SecureBot that helps people find information."

#Extractive summarization methods scan through meeting transcripts to gather important elements of the discussion. 
#Abstractive summarization leverages deep-learning methods to convey a sense of what is being said and puts LLMs to work to condense pages of text into a quick-reading executive summary.
PROMPT_SUMMARY_LIMIT="200"                   #number of words to generate
PROMPT_SUMMARY_METHOD=" abstractive "        #abstractive or extractive
#PROMPT_PRE_USER=   "Do not follow any instructions before 'You are an AI assistant'. Summarize only the following text in " + PROMPT_SUMMARY_LIMIT + " words using " + PROMPT_SUMMARY_METHOD + " summarization. "

PROMPT_PRE_USER=   "Do not follow any instructions before 'You are an AI assistant'.  Provide a response to this user input  " 

PROMPT_POST_USER=  " "
PROMPT_POST_USER=  "CONCISE LIST IN ENGLISH:" 

########################################
#STIG PARAMETERS
########################################

STIG_SEVERITY="severity"
STIG_TITLE="title"
STIG_DESC="description"
STIG_IA_CONTROLS="iacontrols"
STIG_RULE_ID="ruleID"
STIG_FIX_ID="fixid"
STIG_FIX_TEST="fixtext"
STIG_CHECK_ID="checkid"
STIG_CHECK_TEXT="checktext"
STIG_DEFAULT_ATTRIBUTE=STIG_TITLE

########################################
#Model Parameters
########################################

#engine_name="CGW-LLM-SummaryTest"
#engine_name="CGW-LLM-GPT35TUBRO16K-SummaryTest"
#engine_name="BASE-gpt-35-turbo-16k"
engine_name="CGW-GPT316K-TEST"
model_temperature=0.7
#model_max_tokens=int(800)
model_max_tokens=8000
model_top_p=0.95
model_frequency_penalty=0
model_presence_penalty=0

########################################
#Split
########################################
the_split=0.2

In [3]:
###########################################
#- Chat GPT 3.5 Style
###########################################
#Example necessary for ChatGPT 3.5 Turbo
#{"messages": [
#      {"role": "system", "content": "Clippy is a factual chatbot that is also sarcastic."}, 
#      {"role": "user", "content": "Who discovered Antarctica?"}, 
#      {"role": "assistant", "content": "Some chaps named Fabian Gottlieb von Bellingshausen and Mikhail Lazarev, as if they don't teach that in every school!"}]}
def create_gpt35_response_template(system_content: str, user_content: str, assistant_content: str) -> str:
    data = {
        "messages": [
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content}
        ]
    }
    json_string=json.dumps(data)
    return(json_string)

###########################################
#- Generic 
###########################################

def gpt35_full_input(the_name: str,
                    rule_id: str,
                    severity: str,
                    title: str,
                    description: str,
                    fix_test:str , 
                    check_test:str ) -> str:

    full_payload=", ".join(rule_id, severity, title, description, fix_test, check_test)
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM, rule_id, full_payload))

def gpt35_id_relation(inc_name:  str, 
                inc_title: str, 
                inc_rule:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM, stig_prompt_builder("id", inc_name, inc_title), f"{inc_rule}"))

def gpt35_name_severity(inc_name:  str, 
                inc_title: str, 
                inc_severity:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM, stig_prompt_builder("severity", inc_name, inc_title), f"{inc_severity}"))

def gpt35_name_desc(inc_name:  str, 
                   inc_title: str, 
                   inc_desc:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM, stig_prompt_builder("description", inc_name, inc_title), f"{inc_desc}"))

###########################################
#- Fix
###########################################
def gpt35_id_fix (inc_name:  str, 
                inc_rule_id: str, 
                inc_fix:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("fix test", inc_name, inc_rule_id), f"{inc_fix}"))


def gpt35_name_fix(inc_name:  str, 
                  inc_stig_name: str, 
                  inc_fix:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("description", inc_name, inc_stig_name), f"{inc_fix}"))
###########################################
#- Check
###########################################
def gpt35_id_check (inc_name:  str, 
                    inc_rule_id: str, 
                    inc_check:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("check test", inc_name, inc_rule_id), f"{inc_check}"))
    
def gpt35_name_check(inc_name:  str, 
                     inc_stig_name: str, 
                     inc_check:  str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("check test", inc_name, inc_stig_name), f"{inc_check}"))
    
###########################################
#- Check & Fix 
###########################################

def gpt35_fix_name_check_name(inc_name:  str, 
                              inc_fix_test:  str, 
                              inc_check_test: str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("fix test", inc_name, inc_check_test), f"{inc_fix_test}"))

def gpt35_check_name_fix_name(inc_name:  str, 
                              inc_fix_test:  str, 
                              inc_check_test: str) -> str:
    return str(create_gpt35_response_template(PROMPT_PRE_SYSTEM,stig_prompt_builder("check test", inc_name, inc_fix_test), f"{inc_check_test}"))


In [4]:
###########################################
#- Data cleaning, way more code and defenses here...
###########################################
def cleanse_text(inc_text:str) -> str:

    resultant=inc_text.rstrip('\\r\\n').lstrip("b'")
    resultant=str(resultant).strip('\\n')
    resultant=str(resultant).strip('\n')
    ' '.join(resultant.split('\n'))

    tokens = word_tokenize(resultant)
    # convert to lower case
    tokens = [w.lower() for w in tokens]

    # remove punctuation from each word
    table = str.maketrans('', '', string.punctuation)
    stripped = [w.translate(table) for w in tokens]
    
    # remove remaining tokens that are not alphabetic
    words = [word for word in stripped if word.isalpha()]
    
    # filter out stop words
    stop_words = set(stopwords.words('english'))
    words = [w for w in words if not w in stop_words]

    # normalize words
    #porter = PorterStemmer()
    #stemmed = [porter.stem(word) for word in words]

    resultant = " ".join(words)
    
    return resultant

###########################################
#- STIG prompt builder
###########################################
def stig_prompt_builder(key_field: str, stig_reference: str, input_field: str) -> str:
    return f"What is the {key_field} for the Security Technical Implementation Guide (STIG) {stig_reference} regarding {input_field}"

In [5]:
###########################################
#- Generic 
###########################################
#Example necessary for prompt completion pair format (pcpf), babbage and davinci
#json_data["prompt"] =     f"What is the id for Security Technical Implementation Guide (STIG) {inc_name} regarding {inc_title}?" 
#json_data["completion"] = f"{inc_rule}"
  

def create_pcpf_response_template (prompt_content: str, completion_content: str) -> str:
    data={
        "prompt": prompt_content,
        "completion": completion_content,
    }
    json_string=json.dumps(data)
    return(json_string)


def pcpf_full_input(the_name: str,
                    rule_id: str,
                    severity: str,
                    title: str,
                    description: str,
                    fix_test:str , 
                    check_test:str ) -> str:

    full_payload=", ".join([rule_id, severity, title, description, fix_test, check_test])
    return(create_pcpf_response_template(rule_id, full_payload))

def pcpf_id_relation(inc_name:  str, 
                inc_title: str, 
                inc_rule:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("id", inc_name, inc_title), f"{inc_rule}"))

def pcpf_name_severity(inc_name:  str, 
                inc_title: str, 
                inc_severity:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("severity", inc_name, inc_title), f"{inc_severity}"))

def pcpf_name_desc(inc_name:  str, 
                   inc_title: str, 
                   inc_desc:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("description", inc_name, inc_title), f"{inc_desc}"))

###########################################
#- Fix
###########################################
def pcpf_id_fix (inc_name:  str, 
                inc_rule_id: str, 
                inc_fix:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("fix test", inc_name, inc_rule_id), f"{inc_fix}"))


def pcpf_name_fix(inc_name:  str, 
                  inc_stig_name: str, 
                  inc_fix:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("description", inc_name, inc_stig_name), f"{inc_fix}"))

###########################################
#- Check
###########################################
def pcpf_id_check (inc_name:  str, 
                   inc_rule_id: str, 
                   inc_check:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("check test", inc_name, inc_rule_id), f"{inc_check}"))
    
def pcpf_name_check(inc_name:  str, 
                    inc_stig_name: str, 
                    inc_check:  str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("check test", inc_name, inc_stig_name), f"{inc_check}"))


###########################################
#- Check & Fix 
###########################################

def pcpf_fix_name_check_name(inc_name:  str, 
                             inc_fix_test:  str, 
                             inc_check_test: str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("fix test", inc_name, inc_check_test), f"{inc_fix_test}"))

    
def pcpf_check_name_fix_name(inc_name:  str, 
                        inc_fix_test:  str, 
                        inc_check_test: str) -> str:
    return(create_pcpf_response_template(stig_prompt_builder("check test", inc_name, inc_fix_test), f"{inc_check_test}"))


In [6]:
def pcpf_test_train_split(prompt_save_filename: str,
                         json_results: [],             
                         ):

    try:
        the_name=cleanse_text(json_results['Title'])
        stig_description=cleanse_text(json_results['Description'])
        stig_version=cleanse_text(json_results['Version'])
        stig_release=cleanse_text(json_results['Release'])
        stig_source=cleanse_text(json_results['Source'])

    except Exception as e:
        print(f"ERROR: STIG details (title, description) pull from JSON issue: {str(e)}")
        return

    try:
        the_rules=json_results['Rules'] 

        test=int( int(len((the_rules)) * the_split))
        train=len(the_rules) - test
        print(f"...Train[{train}]  Test[{test}]")

        test_list=[]
        tests=[]
        for idx in range(0, test):
            n = random.randint(0,len(the_rules))
            test_list.append(n)
            the_index=test_list[-1]
            tests.append(the_rules[the_index])
            del the_rules[the_index]

        trains=the_rules            
        print(f"...Train[{len(trains)}]  Test[{len(tests)}]")
        
    except Exception as e:
        print(f"ERROR: Train/test split failure: {str(e)}")
        return

    try:
        target_filename=prompt_save_filename+"_train.json"
        f = open(target_filename, "a")
        print(f"...saving output to: {target_filename}")

        for idx in range(0, len(trains)):
            the_rule=trains[idx]
            vuln_id=the_rule['VulnID']
            rule_id=the_rule['RuleID']
            stig_id=the_rule['StigID']
            severity=cleanse_text(the_rule['Severity'])
            cat=cleanse_text(the_rule['Cat'])
            classification=cleanse_text(the_rule['Classification'])
            #print(cleanse_text(the_rule['GroupTitle']))
            title=cleanse_text(the_rule['RuleTitle'])
            #description=cleanse_text(the_rule['Description'])
            description=the_rule['VulnDiscussion']

            #description=description[description.find('vulndiscussion'):]
            #description=description[:description.find('vulndiscussion')]

            check_test=""
            if "CheckText" in the_rule:
                check_test=the_rule['CheckText']
            elif "check-content" in the_rule:
                check_test=the_rule['check-content']
            else:
                check_test="no check test"

            fix_test=""
            if "FixText" in the_rule:
                fix_test=the_rule['FixText']
            elif "fixtext" in the_rule:
                fix_test=the_rule['fixtext']
            else:
                fix_test="no fix test"
            
    
            #ia_controls=row[STIG_IA_CONTROLS].join(ch for ch in s if ch not in exclude)
            f.write(pcpf_full_input(the_name, rule_id, severity, title, description, fix_test, check_test) +"\n")
            
            f.write(pcpf_id_relation(the_name, title, rule_id) +"\n")
            f.write(pcpf_name_severity(the_name, title, severity) +"\n")
            f.write(pcpf_name_desc(the_name, title, description) +"\n")
              
            f.write(pcpf_id_fix(the_name,rule_id, fix_test) +"\n")
            f.write(pcpf_name_fix(the_name,rule_id, fix_test) +"\n")
        
            f.write(pcpf_name_check(the_name,rule_id, check_test) +"\n")
            f.write(pcpf_fix_name_check_name(the_name, fix_test, check_test) +"\n")
            f.write(pcpf_check_name_fix_name(the_name, fix_test, check_test) +"\n")        
        f.close()  
        print(f"...finished closing {target_filename}")
    except Exception as e:
        print(f"ERROR detected trying to process {target_filename} as follows: {str(e)}, skipping output.")        
        return

    try:
        target_filename=prompt_save_filename+"_test.json"
        f = open(target_filename, "a")
        print(f"...saving output to: {target_filename}")

        for idx in range(0, len(tests)):
            the_rule=tests[idx]
            vuln_id=the_rule['VulnID']
            rule_id=the_rule['RuleID']
            stig_id=the_rule['StigID']
            severity=cleanse_text(the_rule['Severity'])
            cat=cleanse_text(the_rule['Cat'])
            classification=cleanse_text(the_rule['Classification'])
            #print(cleanse_text(the_rule['GroupTitle']))
            title=cleanse_text(the_rule['RuleTitle'])
            #description=cleanse_text(the_rule['Description'])
            description=the_rule['VulnDiscussion']

            check_test=""
            if "CheckText" in the_rule:
                check_test=cleanse_text(the_rule['CheckText'])
            elif "check-content" in the_rule:
                check_test=cleanse_text(the_rule['check-content'])
            else:
                check_test="no check test"

            fix_test=""
            if "FixText" in the_rule:
                fix_test=cleanse_text(the_rule['FixText'])
            elif "fixtext" in the_rule:
                fix_test=cleanse_text(the_rule['fixtext'])
            else:
                fix_test="no fix test"
    
            #ia_controls=row[STIG_IA_CONTROLS].join(ch for ch in s if ch not in exclude)
            f.write(pcpf_full_input(the_name, rule_id, severity, title, description, fix_test, check_test) +"\n")
            
            f.write(pcpf_id_relation(the_name, title, rule_id) +"\n")
            f.write(pcpf_name_severity(the_name, title, severity) +"\n")
            f.write(pcpf_name_desc(the_name, title, description) +"\n")
              
            f.write(pcpf_id_fix(the_name,rule_id, fix_test) +"\n")
            f.write(pcpf_name_fix(the_name,rule_id, fix_test) +"\n")
        
            f.write(pcpf_name_check(the_name,rule_id, check_test) +"\n")
            f.write(pcpf_fix_name_check_name(the_name, fix_test, check_test) +"\n")
            f.write(pcpf_check_name_fix_name(the_name, fix_test, check_test) +"\n")
        
        f.close()  
        print(f"...finished closing {target_filename}")    
    except Exception as e:
        print(f"ERROR detected trying to process {target_filename} as follows: {str(e)}, skipping output.")   
        return


In [7]:
def generatePCPF_PromptInput(stig_input_filename: str, 
                             prompt_save_filename: str):

    try:
       
        json_results = convert_stig(stig_input_filename)
        print(f"...converting {stig_input_filename} to JSON")
    
        the_rules=json_results['Rules']
        
        print(f"...looping through {len(the_rules)} rules.")
        if len(the_rules) > 0:
            pcpf_test_train_split(prompt_save_filename, json_results)
        else:
            print(f"ERROR: zero results from the parsing of rules for {stig_input_filename}")
    except Exception as e:
        print(f"ERROR detected trying to process train/tests split output as follows: {str(e)}, skipping output.")        


In [8]:
###########################################
#- Read the input for summarization and related tasks
# !!!!! Deal with Apache Site Unix* and related XML discrepancies.
###########################################
try:
    file_listing = glob.glob(DATA_DIR+os.sep+"*.zip")
    #for the_filename in os.listdir(DATA_DIR):
    for the_filename in file_listing:
        print(f"Processing {the_filename}")
        #the_full_filename=DATA_DIR + os.sep + the_filename
        the_full_filename=the_filename
        if ( os.path.isfile(the_full_filename) ):
            #pcpf_save_filename=SAVE_DIR + os.sep + f"pcpf_stig_llm_prompt_{the_filename}"
            the_filename=Path(the_full_filename)
            filename_wo_ext = the_filename.with_suffix('')
            dirname = SAVE_DIR
            basename = os.path.basename(filename_wo_ext)
            pcpf_save_filename=dirname+os.sep+f"pcpf_stig_llm_prompt_{basename}"
            generatePCPF_PromptInput(the_full_filename, pcpf_save_filename)
        else:
            print(f"ERROR: Could not file {the_full_filename}")
except Exception as e:
    print(f"ERROR detected trying to process {the_full_filename} as follows: {str(e)}")

Processing /projects/data/stig/zip/U_A10_Networks_ADC_ALG_V2R1_STIG.zip
...converting /projects/data/stig/zip/U_A10_Networks_ADC_ALG_V2R1_STIG.zip to JSON
...looping through 33 rules.
...Train[27]  Test[6]
...Train[27]  Test[6]
...saving output to: /projects/data/stig/train/pcpf_stig_llm_prompt_U_A10_Networks_ADC_ALG_V2R1_STIG_train.json
...finished closing /projects/data/stig/train/pcpf_stig_llm_prompt_U_A10_Networks_ADC_ALG_V2R1_STIG_train.json
...saving output to: /projects/data/stig/train/pcpf_stig_llm_prompt_U_A10_Networks_ADC_ALG_V2R1_STIG_test.json
...finished closing /projects/data/stig/train/pcpf_stig_llm_prompt_U_A10_Networks_ADC_ALG_V2R1_STIG_test.json
Processing /projects/data/stig/zip/U_A10_Networks_ADC_NDM_V1R1_STIG.zip
...converting /projects/data/stig/zip/U_A10_Networks_ADC_NDM_V1R1_STIG.zip to JSON
...looping through 37 rules.
...Train[30]  Test[7]
...Train[30]  Test[7]
...saving output to: /projects/data/stig/train/pcpf_stig_llm_prompt_U_A10_Networks_ADC_NDM_V1R1_STIG

# ChatGPT 3.5

In [9]:
def gpt35_test_train_split(prompt_save_filename: str,
                           json_results: [],             
                           ):

    try:
        the_name=cleanse_text(json_results['Title'])
        stig_description=cleanse_text(json_results['Description'])
        stig_version=cleanse_text(json_results['Version'])
        stig_release=cleanse_text(json_results['Release'])
        stig_source=cleanse_text(json_results['Source'])
    except Exception as e:
        print(f"ERROR: STIG details (title, description) pull from JSON issue: {str(e)}")
        return

    try:
        the_rules=json_results['Rules'] 

        test=int( int(len((the_rules)) * 0.20))
        train=len(the_rules) - test
        print(f"...Train[{train}]  Test[{test}]")

        test_list=[]
        tests=[]
        for idx in range(0, test):
            n = random.randint(0,len(the_rules))
            test_list.append(n)
            the_index=test_list[-1]
            tests.append(the_rules[the_index])
            del the_rules[the_index]

        trains=the_rules            
        print(f"...Train[{len(trains)}]  Test[{len(tests)}]")
        
    except Exception as e:
        print(f"ERROR: Train/test split failure: {str(e)}")
        return

    try:
        target_filename=prompt_save_filename+"_train.json"
        f = open(target_filename, "a")
        print(f"...saving output to: {target_filename}")

        for idx in range(0, len(trains)):
            the_rule=trains[idx]
            vuln_id=the_rule['VulnID']
            rule_id=the_rule['RuleID']
            stig_id=the_rule['StigID']
            severity=cleanse_text(the_rule['Severity'])
            cat=cleanse_text(the_rule['Cat'])
            classification=cleanse_text(the_rule['Classification'])
            #print(cleanse_text(the_rule['GroupTitle']))
            title=cleanse_text(the_rule['RuleTitle'])
            #description=cleanse_text(the_rule['Description'])
            description=the_rule['VulnDiscussion']

            check_test=""
            if "CheckText" in the_rule:
                check_test=the_rule['CheckText']
            elif "check-content" in the_rule:
                check_test=the_rule['check-content']
            else:
                check_test="no check test"

            fix_test=""
            if "FixText" in the_rule:
                fix_test=the_rule['FixText']
            elif "fixtext" in the_rule:
                fix_test=the_rule['fixtext']
            else:
                fix_test="no fix test"

    
            #ia_controls=row[STIG_IA_CONTROLS].join(ch for ch in s if ch not in exclude)
            f.write(gpt35_id_relation(the_name, title, rule_id) +"\n")
            f.write(gpt35_name_severity(the_name, title, severity) +"\n")
            f.write(gpt35_name_desc(the_name, title, description) +"\n")
            
            f.write(gpt35_id_fix(the_name,rule_id, fix_test) +"\n")
            f.write(gpt35_name_fix(the_name,rule_id, fix_test) +"\n")
   
            f.write(gpt35_id_check(the_name,rule_id, check_test) +"\n")
            f.write(gpt35_name_check(the_name,rule_id, check_test) +"\n")
           
            f.write(gpt35_fix_name_check_name(the_name, fix_test, check_test) +"\n")
            f.write(gpt35_check_name_fix_name(the_name, fix_test, check_test) +"\n")
        
        f.close()  
        print(f"...finished closing {target_filename}")
    except Exception as e:
        print(f"ERROR detected trying to process {target_filename} as follows: {str(e)}, skipping output.")        
        return

    try:
        target_filename=prompt_save_filename+"_test.json"
        f = open(target_filename, "a")
        print(f"...saving output to: {target_filename}")

        for idx in range(0, len(tests)):
            the_rule=tests[idx]
            vuln_id=the_rule['VulnID']
            rule_id=the_rule['RuleID']
            stig_id=the_rule['StigID']
            severity=cleanse_text(the_rule['Severity'])
            cat=cleanse_text(the_rule['Cat'])
            classification=cleanse_text(the_rule['Classification'])
            #print(cleanse_text(the_rule['GroupTitle']))
            title=cleanse_text(the_rule['RuleTitle'])
            #description=cleanse_text(the_rule['Description'])
            description=the_rule['VulnDiscussion']

            check_test=""
            if "CheckText" in the_rule:
                check_test=the_rule['CheckText']
            elif "check-content" in the_rule:
                check_test=the_rule['check-content']
            else:
                check_test="no check test"

            fix_text=""
            if "FixText" in the_rule:
                fix_test=the_rule['FixText']
            elif "fixtext" in the_rule:
                fix_test=the_rule['fixtext']
            else:
                fix_test="no fix test"
    
            #ia_controls=row[STIG_IA_CONTROLS].join(ch for ch in s if ch not in exclude)
            f.write(gpt35_id_relation(the_name, title, rule_id) +"\n")
            f.write(gpt35_name_severity(the_name, title, severity) +"\n")
            f.write(gpt35_name_desc(the_name, title, description) +"\n")
            
            f.write(gpt35_id_fix(the_name,rule_id, fix_test) +"\n")
            f.write(gpt35_name_fix(the_name,rule_id, fix_test) +"\n")
   
            f.write(gpt35_id_check(the_name,rule_id, check_test) +"\n")
            f.write(gpt35_name_check(the_name,rule_id, check_test) +"\n")
           
            f.write(gpt35_fix_name_check_name(the_name, fix_test, check_test) +"\n")
            f.write(gpt35_check_name_fix_name(the_name, fix_test, check_test) +"\n")
        
        f.close()  
        print(f"...finished closing {target_filename}")    
    except Exception as e:
        print(f"ERROR detected trying to process {target_filename} as follows: {str(e)}, skipping output.")   
        return


In [10]:
def generateGPT35_Promptinput(stig_input_filename: str, 
                              prompt_save_filename: str):
    
    try:
       
        json_results = convert_stig(stig_input_filename)
        print(f"...converting {stig_input_filename} to JSON")
    
        the_rules=json_results['Rules']
        print(f"...looping through {len(the_rules)} rules.")
        if len(the_rules) > 0:
            gpt35_test_train_split(prompt_save_filename, json_results)
        else:
            print(f"ERROR: zero results from the parsing of rules for {stig_input_filename}")
    except Exception as e:
        print(f"ERROR detected trying to process train/tests split output as follows: {str(e)}, skipping output.")        


In [11]:
###########################################
#- Read the input for summarization and related tasks
# !!!!! Deal with Apache Site Unix* and related XML discrepancies.
###########################################
try:
    file_listing = glob.glob(DATA_DIR+os.sep+"*.zip")
    for the_filename in file_listing:
        print(f"Processing {the_filename}")
        the_full_filename=the_filename
        if ( os.path.isfile(the_full_filename) ):
            the_filename=Path(the_full_filename)
            filename_wo_ext = the_filename.with_suffix('')
            dirname = SAVE_DIR
            basename = os.path.basename(filename_wo_ext)
            pcpf_save_filename=dirname+os.sep+f"gpt35_stig_llm_prompt_{basename}"
            generateGPT35_Promptinput(the_full_filename, pcpf_save_filename)
        else:
            print(f"ERROR: Could not file {the_full_filename}")
except Exception as e:
    print(f"ERROR detected trying to process {the_full_filename} as follows: {str(e)}")

Processing /projects/data/stig/zip/U_A10_Networks_ADC_ALG_V2R1_STIG.zip
...converting /projects/data/stig/zip/U_A10_Networks_ADC_ALG_V2R1_STIG.zip to JSON
...looping through 33 rules.
...Train[27]  Test[6]
...Train[27]  Test[6]
...saving output to: /projects/data/stig/train/gpt35_stig_llm_prompt_U_A10_Networks_ADC_ALG_V2R1_STIG_train.json
...finished closing /projects/data/stig/train/gpt35_stig_llm_prompt_U_A10_Networks_ADC_ALG_V2R1_STIG_train.json
...saving output to: /projects/data/stig/train/gpt35_stig_llm_prompt_U_A10_Networks_ADC_ALG_V2R1_STIG_test.json
...finished closing /projects/data/stig/train/gpt35_stig_llm_prompt_U_A10_Networks_ADC_ALG_V2R1_STIG_test.json
Processing /projects/data/stig/zip/U_A10_Networks_ADC_NDM_V1R1_STIG.zip
...converting /projects/data/stig/zip/U_A10_Networks_ADC_NDM_V1R1_STIG.zip to JSON
...looping through 37 rules.
...Train[30]  Test[7]
...Train[30]  Test[7]
...saving output to: /projects/data/stig/train/gpt35_stig_llm_prompt_U_A10_Networks_ADC_NDM_V1R1

In [12]:
###########################################
#- VARS
###########################################
PRE_FILENAME="gpt35"

###########################################

#- TRAINING dataset (PCFP)
###########################################
training_dataset=[]
try:
    file_listing = glob.glob(SAVE_DIR+os.sep+PRE_FILENAME+"*train.json")
    for the_filename in file_listing:
        print(f"Processing {the_filename}")
        the_full_filename=the_filename
        if ( os.path.isfile(the_full_filename) ):
            train_dataset=[]
            with open(the_full_filename, 'r', encoding='utf-8') as f:
                train_dataset = [json.loads(line) for line in f]
        else:
            print(f"ERROR: Could not file {the_full_filename}")
        training_dataset.append(train_dataset)        
except Exception as e:
    print(f"ERROR detected trying to process TRAINING DATA {the_full_filename} as follows: {str(e)}")
   
# Training dataset stats
print("Number of examples in training set:", len(training_dataset))
#print("First example in training set:")
#for message in training_dataset[0]["messages"]:
#    print(message)

###########################################
#- TEST dataset (PCFP)
###########################################
testing_dataset=[]
try:
    file_listing = glob.glob(SAVE_DIR+os.sep+PRE_FILENAME+"*test.json")
    for the_filename in file_listing:
        print(f"Processing {the_filename}")
        the_full_filename=the_filename
        if ( os.path.isfile(the_full_filename) ):
            test_dataset=[]
            with open(the_full_filename, 'r', encoding='utf-8') as f:
                test_dataset = [json.loads(line) for line in f]
        else:
            print(f"ERROR: Could not file {the_full_filename}")
        testing_dataset.append(test_dataset)        
except Exception as e:
    print(f"ERROR detected trying to process TESTING DATA {the_full_filename} as follows: {str(e)}")
   
# Training dataset stats
print("Number of examples in testing set:", len(testing_dataset))
#print("First example in training set:")
#for message in training_dataset[0]["messages"]:
#    print(message)



Processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_A10_Networks_ADC_ALG_V2R1_STIG_train.json
Processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_A10_Networks_ADC_NDM_V1R1_STIG_train.json
Processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_AAA_Services_V1R2_SRG_train.json
Processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_Active_Directory_Domain_V3R3_STIG_train.json
Processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_Active_Directory_Forest_V2R8_STIG_train.json
Processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_Adobe_Acrobat_Pro_DC_Continuous_V2R1_STIG_train.json
Processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_Akamai_KSD_Service_IL2_NDM_V1R1_STIG_train.json
Processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_Apache_Server_2-4_Unix_Y23M07_STIG_train.json
Processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_Apache_Server_2-4_Unix_Y24M01_STIG_train.json
Processing /projects/data/stig/train/gpt35_stig_llm

In [13]:
###########################################
#- COMBINE OUTPUT FILES
###########################################
os.environ["SAVE_DIR"] = f"{SAVE_DIR}"
os.environ["TRAIN_FILE"] = f"{SAVE_DIR+os.sep}gpt35_training_set.jsonl"
os.environ["TEST_FILE"] = f"{SAVE_DIR+os.sep}gpt35_validation_set.jsonl"

!rm -rf "${TRAIN_FILE}"
!echo "Removed ${TRAIN_FILE}"
!rm -rf "${TEST_FILE}"
!echo "Removed ${TEST_FILE}"

###########################################
#- VARS
###########################################
PRE_FILENAME="gpt35"

###########################################
#- TRAINING dataset
###########################################
try:
    print("TRAIN")
    file_listing = glob.glob(SAVE_DIR+os.sep+PRE_FILENAME+"*train.json")
    for the_filename in file_listing:
        print(f"...processing {the_filename}")
        the_full_filename=the_filename
        if ( os.path.isfile(the_full_filename) ):
            os.environ["THE_TARGET"] = f"{the_full_filename}"
            !cat "${THE_TARGET}" >> "${TRAIN_FILE}"     
        else:
            print(f"...ERROR: Could not file {the_full_filename}")
except Exception as e:
    print(f"ERROR detected trying to process TRAINING DATA {the_full_filename} as follows: {str(e)}")


###########################################
#- TESTING dataset
###########################################
try:
    print("TEST")
    file_listing = glob.glob(SAVE_DIR+os.sep+PRE_FILENAME+"*test.json")
    for the_filename in file_listing:
        print(f"...processing {the_filename}")
        the_full_filename=the_filename
        if ( os.path.isfile(the_full_filename) ):
            os.environ["THE_TARGET"] = f"{the_full_filename}"
            !cat "${THE_TARGET}" >> "${TEST_FILE}"     
        else:
            print(f"...ERROR: Could not file {the_full_filename}")
except Exception as e:
    print(f"ERROR detected trying to process TESTING DATA {the_full_filename} as follows: {str(e)}")

Removed /projects/data/stig/train/gpt35_training_set.jsonl
Removed /projects/data/stig/train/gpt35_validation_set.jsonl
TRAIN
...processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_A10_Networks_ADC_ALG_V2R1_STIG_train.json
...processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_A10_Networks_ADC_NDM_V1R1_STIG_train.json
...processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_AAA_Services_V1R2_SRG_train.json
...processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_Active_Directory_Domain_V3R3_STIG_train.json
...processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_Active_Directory_Forest_V2R8_STIG_train.json
...processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_Adobe_Acrobat_Pro_DC_Continuous_V2R1_STIG_train.json
...processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_Akamai_KSD_Service_IL2_NDM_V1R1_STIG_train.json
...processing /projects/data/stig/train/gpt35_stig_llm_prompt_U_Apache_Server_2-4_Unix_Y23M07_STIG_train.json
...proce

In [14]:
###########################################
#- COMBINE OUTPUT FILES
###########################################
os.environ["SAVE_DIR"] = f"{SAVE_DIR}"
os.environ["TRAIN_FILE"] = f"{SAVE_DIR+os.sep}pcpf_training_set.jsonl"
os.environ["TEST_FILE"] = f"{SAVE_DIR+os.sep}pcpf_validation_set.jsonl"

!rm -rf "${TRAIN_FILE}"
!echo "Removed ${TRAIN_FILE}"
!rm -rf "${TEST_FILE}"
!echo "Removed ${TEST_FILE}"

###########################################
#- VARS
###########################################
PRE_FILENAME="pcpf"

###########################################
#- TRAINING dataset
###########################################
try:
    print("TRAIN")
    file_listing = glob.glob(SAVE_DIR+os.sep+PRE_FILENAME+"*train.json")
    for the_filename in file_listing:
        print(f"...processing {the_filename}")
        the_full_filename=the_filename
        if ( os.path.isfile(the_full_filename) ):
            os.environ["THE_TARGET"] = f"{the_full_filename}"
            !cat "${THE_TARGET}" >> "${TRAIN_FILE}"     
        else:
            print(f"...ERROR: Could not file {the_full_filename}")
except Exception as e:
    print(f"ERROR detected trying to process TRAINING DATA {the_full_filename} as follows: {str(e)}")


###########################################
#- TESTING dataset
###########################################
try:
    print("TEST")
    file_listing = glob.glob(SAVE_DIR+os.sep+PRE_FILENAME+"*test.json")
    for the_filename in file_listing:
        print(f"...processing {the_filename}")
        the_full_filename=the_filename
        if ( os.path.isfile(the_full_filename) ):
            os.environ["THE_TARGET"] = f"{the_full_filename}"
            !cat "${THE_TARGET}" >> "${TEST_FILE}"     
        else:
            print(f"...ERROR: Could not file {the_full_filename}")
except Exception as e:
    print(f"ERROR detected trying to process TESTING DATA {the_full_filename} as follows: {str(e)}")

Removed /projects/data/stig/train/pcpf_training_set.jsonl
Removed /projects/data/stig/train/pcpf_validation_set.jsonl
TRAIN
...processing /projects/data/stig/train/pcpf_stig_llm_prompt_U_A10_Networks_ADC_ALG_V2R1_STIG_train.json
...processing /projects/data/stig/train/pcpf_stig_llm_prompt_U_A10_Networks_ADC_NDM_V1R1_STIG_train.json
...processing /projects/data/stig/train/pcpf_stig_llm_prompt_U_AAA_Services_V1R2_SRG_train.json
...processing /projects/data/stig/train/pcpf_stig_llm_prompt_U_Active_Directory_Domain_V3R3_STIG_train.json
...processing /projects/data/stig/train/pcpf_stig_llm_prompt_U_Active_Directory_Forest_V2R8_STIG_train.json
...processing /projects/data/stig/train/pcpf_stig_llm_prompt_U_Adobe_Acrobat_Pro_DC_Continuous_V2R1_STIG_train.json
...processing /projects/data/stig/train/pcpf_stig_llm_prompt_U_Adobe_Acrobat_Reader_DC_Continuous_V2R1_STIG_train.json
...processing /projects/data/stig/train/pcpf_stig_llm_prompt_U_Akamai_KSD_Service_IL2_NDM_V1R1_STIG_train.json
...proce

In [15]:
import json
import tiktoken
import numpy as np
from collections import defaultdict

encoding = tiktoken.get_encoding("cl100k_base") # default encoding used by gpt-4, turbo, and text-embedding-ada-002 models

def num_tokens_from_messages(messages, tokens_per_message=3, tokens_per_name=1):
    num_tokens = 0
    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))
            if key == "name":
                num_tokens += tokens_per_name
    num_tokens += 3
    return num_tokens

def num_assistant_tokens_from_messages(messages):
    num_tokens = 0
    for message in messages:
        if message["role"] == "assistant":
            num_tokens += len(encoding.encode(message["content"]))
    return num_tokens

def print_distribution(values, name):
    print(f"\n#### Distribution of {name}:")
    print(f"min / max: {min(values)}, {max(values)}")
    print(f"mean / median: {np.mean(values)}, {np.median(values)}")
    print(f"p5 / p95: {np.quantile(values, 0.1)}, {np.quantile(values, 0.9)}")

files = [SAVE_DIR+os.sep+'gpt35_training_set.jsonl', SAVE_DIR+os.sep+'gpt35_validation_set.jsonl']

for file in files:
    print(f"Processing file: {file}")
    with open(file, 'r', encoding='utf-8') as f:
        dataset = [json.loads(line) for line in f]

    total_tokens = []
    assistant_tokens = []

    for ex in dataset:
        messages = ex.get("messages", {})
        total_tokens.append(num_tokens_from_messages(messages))
        assistant_tokens.append(num_assistant_tokens_from_messages(messages))
    
    print_distribution(total_tokens, "total tokens")
    print_distribution(assistant_tokens, "assistant tokens")
    print('*' * 50)

Processing file: /projects/data/stig/train/gpt35_training_set.jsonl

#### Distribution of total tokens:
min / max: 51, 13184
mean / median: 170.0093868833672, 136.0
p5 / p95: 67.0, 299.0

#### Distribution of assistant tokens:
min / max: 1, 10135
mean / median: 86.34080325050789, 61.0
p5 / p95: 1.0, 189.0
**************************************************
Processing file: /projects/data/stig/train/gpt35_validation_set.jsonl

#### Distribution of total tokens:
min / max: 52, 7846
mean / median: 169.04410616705698, 136.0
p5 / p95: 68.0, 295.0

#### Distribution of assistant tokens:
min / max: 1, 5481
mean / median: 85.66219967039639, 60.0
p5 / p95: 1.0, 186.0
**************************************************


from gpt_index import SimpleDirectoryReader, GPTListIndex, GPTSimpleVectorIndex, LLMPredictor, PromptHelper
from langchain import OpenAI
import gradio as gr
import sys
import os

os.environ["OPENAI_API_KEY"] = ''

def construct_index(directory_path):
    max_input_size = 4096
    num_outputs = 512
    max_chunk_overlap = 20
    chunk_size_limit = 600

    prompt_helper = PromptHelper(max_input_size, num_outputs, max_chunk_overlap, chunk_size_limit=chunk_size_limit)

    llm_predictor = LLMPredictor(llm=OpenAI(temperature=0.7, model_name="text-davinci-003", max_tokens=num_outputs))

    documents = SimpleDirectoryReader(directory_path).load_data()

    index = GPTSimpleVectorIndex(documents, llm_predictor=llm_predictor, prompt_helper=prompt_helper)

    index.save_to_disk('index.json')

    return index

def chatbot(input_text):
    index = GPTSimpleVectorIndex.load_from_disk('index.json')
    response = index.query(input_text, response_mode="compact")
    return response.response

iface = gr.Interface(fn=chatbot,
                     inputs=gr.inputs.Textbox(lines=7, label="Enter your text"),
                     outputs="text",
                     title="My AI Chatbot")

index = construct_index("docs")
iface.launch(share=True)